# BNN demo: PDMP samplers for regression and classification

This notebook runs the two minimal example scripts from `examples/`:

- `bnn_regression.py` — Bayesian regression with a fully-connected BNN
- `bnn_classification.py` — Bayesian multiclass classification with the same architecture family

Both scripts wrap the same underlying pipeline: build an `nn.Module`, construct a `ParamSpec` from it, set up a Gaussian prior + appropriate likelihood, and sample the posterior with one of four PDMP samplers (`zigzag`, `sticky_zigzag`, `boomerang`, `sticky_boomerang`).

The point of this notebook is to verify the pipeline runs end-to-end and to show what the posterior looks like on a couple of toy problems.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.scripts.bnns.bnn_regression import sample_bnn as sample_bnn_regression
from sazz.scripts.bnns.bnn_classification import sample_bnn_classification

torch.set_default_dtype(torch.float64)

## Part 1 — Regression

A 1-D toy: `y = sin(2πx) + 0.1·ε` on `x ∈ [-1, 1]`, with 60 noisy training points. Network is `[1, 50, 1]` with tanh activations.

We expect the posterior predictive band to track the data closely on `[-1, 1]` and grow wider where there are no training points.

In [ ]:
torch.manual_seed(0)
N = 60
x_train = torch.linspace(-1, 1, N).unsqueeze(-1)
y_train = (2 * torch.pi * x_train.squeeze()).sin() + 0.1 * torch.randn(N)

plt.figure(figsize=(7, 3))
plt.scatter(x_train.squeeze(), y_train, s=15, alpha=0.7)
plt.xlabel("x"); plt.ylabel("y"); plt.title("training data")
plt.tight_layout(); plt.show()

In [ ]:
regression_samples = {}
regression_skeletons = {}
regression_target = None

for sampler_name in ["zigzag", "sticky_zigzag", "boomerang", "sticky_boomerang"]:
    print(f"\n=== {sampler_name} ===")
    samples, target, skeletons = sample_bnn_regression(
        x_train, y_train,
        layer_sizes=[1, 50, 1],
        activation="tanh",
        noise_std=0.1,
        prior_std_weight=1.0,
        prior_std_bias=1.0,
        sampler=sampler_name,
        n_skeleton=5_000,
        n_resample=2_000,
        seed=0,
    )
    regression_samples[sampler_name] = samples
    regression_skeletons[sampler_name] = skeletons
    regression_target = target

print("\nAll samplers done.")
print(f"samples shapes: {[(k, tuple(v.shape)) for k, v in regression_samples.items()]}")

In [ ]:
x_grid = torch.linspace(-1.4, 1.4, 200).unsqueeze(-1)
likelihood = regression_target.meta["model"].likelihood

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
y_true_grid = (2 * torch.pi * x_grid.squeeze()).sin()

for ax, name in zip(axes.flat, regression_samples):
    samples = regression_samples[name]
    with torch.no_grad():
        preds = torch.stack([
            likelihood.predict(b, x_grid).squeeze(-1) for b in samples
        ])
    mean = preds.mean(0)
    std  = preds.std(0)

    xg = x_grid.squeeze().numpy()
    ax.fill_between(xg, (mean - 2*std).numpy(), (mean + 2*std).numpy(),
                    alpha=0.25, label="±2σ posterior")
    ax.plot(xg, mean.numpy(), lw=2, label="posterior mean")
    ax.plot(xg, y_true_grid.numpy(), "k--", lw=1, alpha=0.6, label="true f(x)")
    ax.scatter(x_train.squeeze().numpy(), y_train.numpy(),
               s=10, c="red", alpha=0.6, zorder=3, label="train")
    ax.set_title(name); ax.set_xlabel("x")
    if ax is axes[0, 0]: ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
print(f"x_ref norm: {target.x_ref.norm():.4f}")
print(f"x_ref max abs: {target.x_ref.abs().max():.4f}")

# Also: what does the MAP predict?
likelihood = target.meta["model"].likelihood
with torch.no_grad():
    map_preds = likelihood.predict(target.x_ref, x_grid).squeeze(-1)
plt.plot(x_grid.squeeze(), map_preds.numpy(), label="MAP fit")
plt.scatter(x_train.squeeze(), y_train, s=10, alpha=0.5)
plt.legend(); plt.show()

In [ ]:
for name in ["sticky_zigzag", "sticky_boomerang"]:
    samples = regression_samples[name]
    near_zero = (samples.abs() < 1e-8).float().mean().item()
    print(f"{name}: fraction of (sample, coord) pairs at exactly 0 = {near_zero:.2%}")

## Part 2 — Classification

A 2-D toy: three Gaussian blobs centred at the corners of a triangle, 40 points per class. Network is `[2, 16, 3]` with tanh activations.

We expect clean decision boundaries between the three classes and high predictive entropy in regions away from the data (since posterior uncertainty grows there).

In [ ]:
torch.manual_seed(0)
centres = torch.tensor([[ 1.5,  0.0],
                        [-0.75,  1.3],
                        [-0.75, -1.3]])
n_per_class = 40
X_cls = torch.cat([centres[k] + 0.4 * torch.randn(n_per_class, 2)
                   for k in range(3)], dim=0)
y_cls = torch.cat([torch.full((n_per_class,), k) for k in range(3)], dim=0)

plt.figure(figsize=(5, 5))
for k in range(3):
    m = y_cls == k
    plt.scatter(X_cls[m, 0], X_cls[m, 1], label=f"class {k}", s=15, alpha=0.7)
plt.legend(); plt.gca().set_aspect("equal")
plt.title("training data"); plt.tight_layout(); plt.show()

In [ ]:
classification_samples = {}
classification_skeletons = {}
classification_target = None

for sampler_name in ["zigzag", "sticky_zigzag", "boomerang", "sticky_boomerang"]:
    print(f"\n=== {sampler_name} ===")
    samples, target, skeletons = sample_bnn_classification(
        X_cls, y_cls,
        layer_sizes=[2, 16, 3],
        activation="tanh",
        prior_std_weight=1.0,
        prior_std_bias=1.0,
        sampler=sampler_name,
        n_skeleton=5_000,
        n_resample=2_000,
        seed=0,
    )
    classification_samples[sampler_name] = samples
    classification_skeletons[sampler_name] = skeletons
    classification_target = target

print("\nAll samplers done.")

In [ ]:
grid = torch.linspace(-3, 3, 60)
xx, yy = torch.meshgrid(grid, grid, indexing="xy")
X_grid = torch.stack([xx.flatten(), yy.flatten()], dim=-1)

likelihood = classification_target.meta["model"].likelihood

probs_per_sampler = {}
for name, samples in classification_samples.items():
    with torch.no_grad():
        logits = torch.stack([likelihood.predict(b, X_grid) for b in samples])
    probs = torch.softmax(logits, dim=-1).mean(0)
    probs_per_sampler[name] = probs.reshape(grid.shape[0], grid.shape[0], 3)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))

for col, name in enumerate(probs_per_sampler):
    probs_grid = probs_per_sampler[name]                       # [G, G, 3]
    pred_class = probs_grid.argmax(-1).numpy()
    entropy = -(probs_grid * probs_grid.clamp(min=1e-12).log()).sum(-1).numpy()

    # Top row: predicted class
    ax = axes[0, col]
    ax.imshow(pred_class, origin="lower",
              extent=(grid.min(), grid.max(), grid.min(), grid.max()),
              cmap="viridis", alpha=0.6)
    for k in range(3):
        m = y_cls == k
        ax.scatter(X_cls[m, 0], X_cls[m, 1], s=10, edgecolor="k",
                   facecolor=plt.cm.viridis(k / 2), linewidth=0.4)
    ax.set_title(f"{name}\npredicted class")
    ax.set_aspect("equal")

    # Bottom row: predictive entropy
    ax = axes[1, col]
    im = ax.imshow(entropy, origin="lower",
                   extent=(grid.min(), grid.max(), grid.min(), grid.max()),
                   cmap="magma")
    ax.scatter(X_cls[:, 0], X_cls[:, 1], s=4, c="white", alpha=0.6)
    ax.set_title("predictive entropy")
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout(); plt.show()

In [ ]:
for name, samples in classification_samples.items():
    with torch.no_grad():
        logits = torch.stack([likelihood.predict(b, X_cls) for b in samples])
    probs = torch.softmax(logits, dim=-1).mean(0)
    acc = (probs.argmax(-1) == y_cls).float().mean().item()
    print(f"{name}: train accuracy = {acc:.3f}")

## Two moons

In [ ]:
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=200, noise=0.02, random_state=0)
X_moons = torch.tensor(X_np)
y_moons = torch.tensor(y_np, dtype=torch.long)

plt.figure(figsize=(5, 5))
for k in range(2):
    m = y_moons == k
    plt.scatter(X_moons[m, 0], X_moons[m, 1], label=f"class {k}", s=15, alpha=0.7)
plt.legend(); plt.gca().set_aspect("equal")
plt.title("two moons"); plt.tight_layout(); plt.show()

In [ ]:
moons_samples = {}
moons_skeletons = {}
moons_target = None

for sampler_name in ["zigzag", "sticky_zigzag", "boomerang", "sticky_boomerang"]: 
    print(f"\n=== {sampler_name} ===")
    samples, target, skeletons = sample_bnn_classification(
        X_moons, y_moons,
        layer_sizes=[2, 50, 2],          # <-- 2 outputs for binary
        activation="tanh",
        prior_std_weight=1.0,
        prior_std_bias=3.0,
        fan_in_scaling=False,
        prior_inclusion_weight=0.2
        sampler=sampler_name,
        n_skeleton=10_000,
        n_resample=2_000,
        seed=0,
    )
    moons_samples[sampler_name] = samples
    moons_skeletons[sampler_name] = skeletons
    moons_target = target

print("\nAll samplers done.")

In [ ]:
grid_x = torch.linspace(-1.5, 2.5, 80)
grid_y = torch.linspace(-1.0, 1.5, 80)
xx, yy = torch.meshgrid(grid_x, grid_y, indexing="xy")
X_grid = torch.stack([xx.flatten(), yy.flatten()], dim=-1)

likelihood = moons_target.meta["model"].likelihood

probs_per_sampler = {}
for name, samples in moons_samples.items():
    with torch.no_grad():
        logits = torch.stack([likelihood.predict(b, X_grid) for b in samples])
    probs = torch.softmax(logits, dim=-1).mean(0)
    probs_per_sampler[name] = probs.reshape(grid_y.shape[0], grid_x.shape[0], 2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))

for col, name in enumerate(probs_per_sampler):
    probs_grid = probs_per_sampler[name]                        # [G, G, 2]
    pred_class = probs_grid.argmax(-1).numpy()
    entropy = -(probs_grid * probs_grid.clamp(min=1e-12).log()).sum(-1).numpy()

    # Top row: predicted class
    ax = axes[0, col]
    ax.imshow(pred_class, origin="lower",
          extent=(grid_x.min(), grid_x.max(), grid_y.min(), grid_y.max()),
          cmap="coolwarm", alpha=0.5, vmin=0, vmax=1)
    for k in range(2):
        m = y_moons == k
        ax.scatter(X_moons[m, 0], X_moons[m, 1], s=10, edgecolor="k",
                facecolor=plt.cm.viridis(k / 1), linewidth=0.4)   # k/1 not k/2
    ax.set_title(f"{name}\npredicted class")
    ax.set_aspect("equal")

    # Bottom row: predictive entropy
    ax = axes[1, col]
    im = ax.imshow(entropy, origin="lower",
                   extent=(grid_x.min(), grid_x.max(), grid_y.min(), grid_y.max()),
                   cmap="magma")
    ax.scatter(X_moons[:, 0], X_moons[:, 1], s=4, c="white", alpha=0.6)
    ax.set_title("predictive entropy")
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout(); plt.show()

In [ ]:
for name, samples in moons_samples.items():
    with torch.no_grad():
        logits = torch.stack([likelihood.predict(b, X_moons) for b in samples])
    probs = torch.softmax(logits, dim=-1).mean(0)
    acc = (probs.argmax(-1) == y_moons).float().mean().item()
    print(f"{name}: train accuracy = {acc:.3f}")

In [ ]:
likelihood = moons_target.meta["model"].likelihood
with torch.no_grad():
    map_logits = likelihood.predict(moons_target.x_ref, X_grid)
    map_probs = torch.softmax(map_logits, dim=-1)
plt.imshow(map_probs[..., 1].reshape(80, 80).numpy(), origin="lower",
           extent=(grid_x.min(), grid_x.max(), grid_y.min(), grid_y.max()))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, edgecolor='k')
plt.title("MAP class-1 probability")
plt.show()